# Local-model answers for the compression + routing study (Kaggle T4)

1. Add the dataset containing `prompts.jsonl` (made by `backend/scripts/bench/kaggle_prepare.py`).
2. Settings: **GPU T4 x2**, Internet **on** (to download the model).
3. Run all. Output: `/kaggle/working/answers.jsonl` (resumable — re-run to continue after a timeout).
4. Download `answers.jsonl` into `backend/data/bench/kaggle/` and run
   `python backend/scripts/bench/kaggle_import.py`.


In [ ]:
import os, subprocess, sys
try:
    import vllm  # noqa
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], check=False)


In [ ]:
import glob, json, os, time
MODEL = "Qwen/Qwen2.5-3B-Instruct"   # Apache-2.0; weaker 'cheap' model, different family from gpt-oss
MAX_NEW = 512
CHUNK = 2000
src = glob.glob("/kaggle/input/**/prompts.jsonl", recursive=True)
assert src, "attach the dataset that contains prompts.jsonl"
OUT = "/kaggle/working/answers.jsonl"
prev = glob.glob("/kaggle/input/**/prev_answers.jsonl", recursive=True)
if prev and not os.path.exists(OUT):  # resume from an earlier Kaggle run
    import shutil; shutil.copy(prev[0], OUT)
rows = [json.loads(l) for l in open(src[0], encoding="utf-8")]
done = set()
if os.path.exists(OUT):
    for l in open(OUT, encoding="utf-8"):
        try: done.add(json.loads(l)["key"])
        except Exception: pass
todo = [r for r in rows if r["key"] not in done]
print(f"{len(rows)} prompts, {len(done)} done, {len(todo)} to go")


In [ ]:
import torch
n_gpu = torch.cuda.device_count()
use_vllm = True
try:
    from vllm import LLM, SamplingParams
    llm = LLM(model=MODEL, dtype="half", tensor_parallel_size=max(1, n_gpu), max_model_len=4096,
              gpu_memory_utilization=0.90, seed=0)
    sp = SamplingParams(temperature=0.0, max_tokens=MAX_NEW, seed=0)
    tok = llm.get_tokenizer()
except Exception as e:
    print("vLLM unavailable, falling back to transformers:", e)
    use_vllm = False
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="auto")

def chat_text(r):
    msgs = [{"role": "system", "content": r["system"]}, {"role": "user", "content": r["prompt"]}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def generate(batch):
    texts = [chat_text(r) for r in batch]
    if use_vllm:
        outs = llm.generate(texts, sp)
        return [(o.outputs[0].text, len(o.prompt_token_ids), len(o.outputs[0].token_ids)) for o in outs]
    res = []
    for i in range(0, len(texts), 16):
        enc = tok(texts[i:i+16], return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            g = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False)
        for j in range(g.shape[0]):
            new = g[j, enc["input_ids"].shape[1]:]
            res.append((tok.decode(new, skip_special_tokens=True), int(enc["attention_mask"][j].sum()),
                        int((new != tok.pad_token_id).sum())))
    return res


In [ ]:
t0 = time.time()
with open(OUT, "a", encoding="utf-8") as fh:
    for s in range(0, len(todo), CHUNK):
        batch = todo[s:s+CHUNK]
        for r, (text, p_tok, c_tok) in zip(batch, generate(batch)):
            fh.write(json.dumps({"key": r["key"], "item_id": r["item_id"], "lang": r["lang"], "methods": r["methods"],
                                 "model": MODEL, "content": text, "prompt_tokens": p_tok,
                                 "completion_tokens": c_tok}, ensure_ascii=False) + "\n")
        fh.flush()
        print(f"{s+len(batch)}/{len(todo)} done, {time.time()-t0:.0f}s")
print("finished ->", OUT)
